In [ ]:
pip install sklearn-crfsuite

In [2]:
import re
import pandas as pd

In [3]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
def tokenise_with_alignment(text, tokenizer):
    words = text.strip().split()

    subword_ids = []
    tokens = []
    word_first_subword = []

    current_index = 0

    for word in words:
        pieces = tokenizer.tokenize(word)

        if not pieces:
            pieces = [tokenizer.unk_token]

        word_first_subword.append(current_index)

        tokens.extend(pieces)
        subword_ids.extend(tokenizer.convert_tokens_to_ids(pieces))

        current_index += len(pieces)

    return words, tokens, subword_ids, word_first_subword

In [6]:
train_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/train.parquet")
val_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/val.parquet")
test_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/test.parquet")

In [ ]:
def prepare_dataset(df):
    all_sentences = []
    
    for i in range(len(df)):
        words, tokens, _, word_first_subword = tokenise_with_alignment(
            df["unmasked_text"].iloc[i],
            tokenizer
        )
    
        subword_labels = df["token_entity_labels"].iloc[i]
    
        sentence_words = []
        sentence_labels = []
    
        for w, idx in zip(words, word_first_subword):
            if idx < len(subword_labels):
                label = subword_labels[idx]
            else:
                label = "O"
    
            sentence_words.append(w)
            sentence_labels.append(label)
    
        all_sentences.append([sentence_words, sentence_labels])

    return pd.DataFrame(all_sentences, columns=["words", "labels"])

In [8]:
train_df=prepare_dataset(train_df)

In [9]:
val_df=prepare_dataset(val_df)

In [10]:
test_df=prepare_dataset(test_df)

In [ ]:
train_sentences = []
for _, row in train_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    train_sentences.append(sentence)

In [ ]:
val_sentences = []
for _, row in val_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    val_sentences.append(sentence)

In [ ]:
test_sentences = []
for _, row in test_df.iterrows():
    token_entity_labels = row["labels"]
    tokenised_unmasked_text = row["words"]
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    test_sentences.append(sentence)

In [ ]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word_lower": word.lower(),
        "word_suffix_3": word[-3:],
        "word_suffix_2": word[-2:],
        "word_isupper": word.isupper(),
        "word_istitle": word.istitle(),
        "word_isdigit": word.isdigit(),
    }

    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "prev_word_lower": prev_word.lower(),
            "prev_word_istitle": prev_word.istitle(),
            "prev_word_isupper": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  
        
    
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "next_word_lower": next_word.lower(),
            "next_word_istitle": next_word.istitle(),
            "next_word_isupper": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [15]:
def extract_features(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [16]:
X_train1, y_train1 = extract_features(train_sentences)
X_val1, y_val1 = extract_features(val_sentences)
X_test1, y_test1 = extract_features(test_sentences)

In [17]:
crf1 = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf1.fit(X_train1, y_train1)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

In [18]:
y_pred = crf1.predict(X_val1)

print(metrics.flat_classification_report(
    y_val1, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.978     1.000     0.989        87
   B-ACCOUNTNUMBER      0.685     0.485     0.568       103
B-CREDITCARDNUMBER      0.623     0.447     0.521        85
           B-EMAIL      0.947     0.984     0.966       128
            B-IPV4      0.588     0.472     0.524       106
            B-IPV6      0.684     0.299     0.416        87
             B-MAC      0.935     0.430     0.589       100
        B-PASSWORD      0.891     0.527     0.662        93
    B-PHONE_NUMBER      0.947     0.807     0.871        88
             B-SSN      0.841     0.569     0.679        65
        B-USERNAME      0.954     0.466     0.626       133
     I-ACCOUNTNAME      0.974     0.974     0.974       153
    I-PHONE_NUMBER      0.946     0.957     0.951        92
             I-SSN      0.972     0.921     0.946        38
                 O      0.993     0.998     0.996     57818

          accuracy                    

In [19]:
from joblib import dump, load
dump(crf1, "crf_model1.joblib")

['crf_model1.joblib']

# additional features

In [26]:
def get_word_shape(word):
    shape = []
    for c in word:
        if c.isupper():
            shape.append("X")
        elif c.islower():
            shape.append("x")
        elif c.isdigit():
            shape.append("d")
        else:
            shape.append(c)
    return "".join(shape)

In [ ]:
def word2features2(sentence, i):
    word = sentence[i][0]
    
    has_any_digit =sum([c.isdigit() for c in word])>0
    has_any_lower =sum([c.islower() for c in word])>0
    has_any_upper = sum([c.isupper() for c in word])>0
    has_any_special_char = sum([not c.isalnum() for c in word])>0
    number_of_digits = sum([c.isdigit() for c in word])
    number_of_alphabetical_characters = sum([c.isalpha() for c in word])
    number_of_special_characters = sum([not c.isalnum() for c in word])
    have_any_dot= "." in word
    have_any_colon = ":" in word
    have_any_dash = "-" in word
    have_any_slash = "/" in word
    have_any_at = "@" in word
    is_email=bool(re.fullmatch(r".+@.+\..+", word)) 
    is_ipv4= bool(re.fullmatch(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", word))
    is_mac= bool(re.fullmatch(r"([0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}", word))
    is_hex= bool(re.fullmatch(r"[0-9a-fA-F]{8,}", word))
    is_four_digit= bool(re.fullmatch(r"\d{4}", word))
    features = {"bias":1.0}
    
    if i > 0:
        prev_word = sentence[i - 1][0]
        prev_has_any_digit = sum([c.isdigit() for c in prev_word])>0
        prev_features={
            "prev word_lower": prev_word.lower(),
            "prev word_shape":get_word_shape(prev_word),
            "prev word_len":len(prev_word),
            "prev istitle": prev_word.istitle(),
            "prev isupper":prev_word.isupper(),
            "prev has_digit": prev_has_any_digit,
            "prev_current": prev_word.lower()+"_"+word.lower(),
        }

    else:
        features["BOS"] = True

    features.update({
        "word lower": word.lower(),
        "word first two char": word[:2],
        "word first three char":word[:3],
        "word last two char":word[-2:],
        "word last three char": word[-3:],
        "word isupper": word.isupper(),
        "word istitle": word.istitle(),
        "word islower": word.islower(),
        "word len": len(word),
        "word shape": get_word_shape(word),
        "word has any digit":has_any_digit,
        "word has any lower":has_any_lower,
        "word has any upper":has_any_upper,
        "word has any special":has_any_special_char,
        "digit count":number_of_digits,
        "alphabetical count":number_of_alphabetical_characters,
        "special chars count":number_of_special_characters,
        "word has_dot": have_any_dot,
        "word_has_colon": have_any_colon,
        "word has_dash":have_any_dash,
        "word has_slash":have_any_slash,
        "word has_at":have_any_at,
        "email":is_email,
        "ipv4":is_ipv4,
        "mac": is_mac,
        "hex": is_hex,
        "is_four_digits": is_four_digit,
    })

    if i < len(sentence) - 1:
        next_word = sentence[i + 1][0]
        next_has_any_digits=sum([c.isdigit() for c in next_word])>0
        next_features={
            "next word_lower":next_word.lower(),
            "next word_shape":get_word_shape(next_word),
            "next word_len":len(next_word),
            "next word_istitle":next_word.istitle(),
            "next word_isupper":next_word.isupper(),
            "next has_digit":next_has_any_digits,
            "current_next": word.lower()+"_"+next_word.lower(),
        }

    else:
        features["EOS"] = True

    if i>0:
        features.update(prev_features)
    if i < len(sentence) - 1:
        features.update(next_features)

    return features

In [ ]:
def extract_features2(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features2(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [ ]:
X_train3, y_train3 = extract_features2(train_sentences)
X_val3, y_val3 = extract_features2(val_sentences)
X_test3, y_test3 = extract_features2(test_sentences)

In [48]:
crf3 = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=150,
    all_possible_transitions=True
)

crf3.fit(X_train3, y_train3)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=150)

In [ ]:
y_pred3 = crf3.predict(X_val3)
print(metrics.flat_classification_report(y_val3, y_pred3, digits=3))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.978     1.000     0.989        87
   B-ACCOUNTNUMBER      1.000     1.000     1.000       103
B-CREDITCARDNUMBER      0.735     0.718     0.726        85
           B-EMAIL      0.992     1.000     0.996       128
            B-IPV4      0.752     0.858     0.802       106
            B-IPV6      0.784     0.793     0.789        87
             B-MAC      0.990     0.980     0.985       100
        B-PASSWORD      0.989     0.935     0.961        93
    B-PHONE_NUMBER      0.989     1.000     0.994        88
             B-SSN      0.985     1.000     0.992        65
        B-USERNAME      0.991     0.842     0.911       133
     I-ACCOUNTNAME      0.968     0.974     0.971       153
    I-PHONE_NUMBER      1.000     1.000     1.000        92
             I-SSN      1.000     1.000     1.000        38
                 O      0.998     0.999     0.999     57818

          accuracy                    

# test

In [ ]:
y_pred = crf3.predict(X_test3)
print(metrics.flat_classification_report(y_test3, y_pred, digits=3))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.990     1.000     0.995       102
   B-ACCOUNTNUMBER      1.000     0.991     0.995       110
B-CREDITCARDNUMBER      0.699     0.730     0.714        89
           B-EMAIL      0.987     1.000     0.994       153
            B-IPV4      0.704     0.833     0.763       120
            B-IPV6      0.750     0.800     0.774       105
             B-MAC      1.000     0.974     0.987        77
        B-PASSWORD      0.979     0.931     0.954       101
    B-PHONE_NUMBER      1.000     1.000     1.000       106
             B-SSN      0.989     0.978     0.983        89
        B-USERNAME      0.969     0.862     0.912       145
     I-ACCOUNTNAME      0.978     1.000     0.989       179
    I-PHONE_NUMBER      1.000     1.000     1.000       108
             I-SSN      1.000     0.958     0.979        48
                 O      0.998     0.998     0.998     63885

          accuracy                    

In [ ]:
from joblib import dump, load
dump(crf3, "crf_model2.joblib")

['crf_model3_cleaning.joblib']